In [1]:
import pandas as pd
import numpy as np
from typing import List, Any
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
import sys

sys.path.insert(0, "..")
warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")

from scipy import stats
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error,
)

from src.data_loader import DataLoader
from src.preprocessing import DataPreprocessor
from src.feature_engineering import FeatureEngineer
from src.model import ModelTrainer
from src.evaluate import ModelEvaluator

print("✅ Imports complete")

✅ Imports complete


In [2]:
# Load Model & Test Data
loader       = DataLoader("../config/config.yaml")
preprocessor = DataPreprocessor("../config/config.yaml")
engineer     = FeatureEngineer()
trainer      = ModelTrainer("../config/config.yaml")
evaluator    = ModelEvaluator()

# Load data
df = loader.load_data()
X, y = loader.split_features_target(df)
X = engineer.run_all(X, y)
df_full = X.copy()
df_full["SalePrice"] = y.values
X_train, X_test, y_train, y_test = preprocessor.run_pipeline(df_full)

# Load trained model
try:
    model = trainer.load_model("../models/best_model.pkl")
    print("✅ Trained model loaded from disk.")
except FileNotFoundError:
    print("⚠️ Model not found. Training now...")
    results = trainer.train_baseline_models(X_train, y_train, X_test, y_test)
    _, model = trainer.select_best_model(results)
    trainer.save_model(model)

INFO:src.data_loader:✅ Config loaded from: ../config/config.yaml


INFO:src.data_loader:✅ DataLoader initialized successfully.
INFO:src.preprocessing:✅ DataPreprocessor initialized.
INFO:src.feature_engineering:✅ FeatureEngineer initialized.
INFO:src.model:✅ Initialized 7 models: ['linear_regression', 'ridge', 'lasso', 'decision_tree', 'random_forest', 'gradient_boosting', 'xgboost']
INFO:src.model:✅ ModelTrainer initialized.
INFO:src.evaluate:✅ ModelEvaluator initialized.
INFO:src.data_loader:🔄 Loading data from: d:\STUDY\Programming\project\house-price-prediction\data\raw\housing_data.csv
INFO:src.data_loader:✅ Data loaded successfully. Shape: (90000, 28)
INFO:src.data_loader:✅ Split complete → X: (90000, 27), y: (90000,)
INFO:src.feature_engineering:==================================================
INFO:src.feature_engineering:🚀 RUNNING FEATURE ENGINEERING PIPELINE
INFO:src.feature_engineering:==================================================
INFO:src.feature_engineering:🔄 Adding age features...
INFO:src.feature_engineering:✅ Age features added.


✅ Trained model loaded from disk.


In [3]:

# Core Metrics
y_pred = model.predict(X_test)
y_pred = np.clip(y_pred, 0, None)

metrics = evaluator.compute_metrics(y_test, y_pred, "XGBoost")



  📊 EVALUATION REPORT: XGBOOST
  R² Score         : 0.9102
  Adjusted R²      : 0.9102
  RMSE             : $   25,174.48
  MAE              : $   18,440.74
  MAPE             :        6.32%
  ✅ PASS: R² score meets threshold (≥ 0.85)


In [4]:
# Actual vs Predicted Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Actual vs Predicted House Prices",
             fontsize=14, fontweight="bold")

# Scatter plot
ax1 = axes[0]
ax1.scatter(y_test, y_pred, alpha=0.4, color="#2563EB", s=25)
lims = [
    min(y_test.min(), y_pred.min()),
    max(y_test.max(), y_pred.max())
]
ax1.plot(lims, lims, "r--", lw=2, label="Perfect Prediction")
ax1.set_xlabel("Actual Price ($)")
ax1.set_ylabel("Predicted Price ($)")
ax1.set_title("Actual vs Predicted")
ax1.legend()

r2 = r2_score(y_test, y_pred)
ax1.text(0.05, 0.93,
         f"R² = {r2:.4f}",
         transform=ax1.transAxes,
         fontsize=11,
         bbox=dict(boxstyle="round", facecolor="#EFF6FF", alpha=0.9))

# Residual plot
ax2 = axes[1]
residuals = y_test - y_pred
ax2.scatter(y_pred, residuals, alpha=0.4, color="#10B981", s=25)
ax2.axhline(y=0, color="red", linestyle="--", lw=2)
ax2.set_xlabel("Predicted Price ($)")
ax2.set_ylabel("Residuals ($)")
ax2.set_title("Residual Plot")

plt.tight_layout()
plt.savefig("../reports/figures/prediction_vs_actual.png", dpi=150)
plt.show()


In [5]:
# Residual Analysis
fig = plt.figure(figsize=(14, 10))
gs  = gridspec.GridSpec(2, 2, figure=fig)
fig.suptitle("Comprehensive Residual Analysis",
             fontsize=14, fontweight="bold")
# Histogram
ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(residuals, bins=50, color="#2563EB",
         alpha=0.7, edgecolor="white")
ax1.axvline(x=0, color="red", linestyle="--")
ax1.set_title("Residual Distribution")
ax1.set_xlabel("Residual ($)")
ax1.set_ylabel("Frequency")
ax1.text(0.05, 0.92,
         f"Mean: ${residuals.mean():,.0f}\nStd: ${residuals.std():,.0f}",
         transform=ax1.transAxes, fontsize=9,
         bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

# Q-Q Plot
ax2 = fig.add_subplot(gs[0, 1])
stats.probplot(residuals, dist="norm", plot=ax2)
ax2.set_title("Q-Q Plot (Normality Test)")

# Residuals vs Predicted
ax3 = fig.add_subplot(gs[1, 0])
ax3.scatter(y_pred, residuals, alpha=0.4, color="#F59E0B", s=20)
ax3.axhline(y=0, color="red", linestyle="--")
ax3.set_xlabel("Predicted Values")
ax3.set_ylabel("Residuals")
ax3.set_title("Residuals vs Predicted")

# Scale-Location Plot
ax4 = fig.add_subplot(gs[1, 1])
sqrt_residuals = np.sqrt(np.abs(residuals))
ax4.scatter(y_pred, sqrt_residuals, alpha=0.4, color="#EF4444", s=20)
ax4.set_xlabel("Predicted Values")
ax4.set_ylabel("√|Residuals|")
ax4.set_title("Scale-Location Plot")

plt.tight_layout()
plt.savefig("../reports/figures/residual_analysis_full.png", dpi=150)
plt.show()

In [6]:

# Error Distribution by Price Range
price_bins = pd.cut(y_test, bins=5, labels=[
    "Very Low", "Low", "Medium", "High", "Very High"
])

error_by_range = pd.DataFrame({
    "PriceRange"    : price_bins,
    "AbsError"      : np.abs(residuals),
    "PctError"      : np.abs(residuals) / y_test * 100,
}).groupby("PriceRange").agg(
    MeanAbsError=("AbsError", "mean"),
    MeanPctError=("PctError", "mean"),
    Count=("AbsError", "count"),
).reset_index()

print("\n📊 Error Analysis by Price Range:")
print(error_by_range.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Error Analysis by Price Range",
             fontsize=13, fontweight="bold")

colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(error_by_range)))[::-1]
axes[0].bar(error_by_range["PriceRange"],
            error_by_range["MeanAbsError"], color=colors)
axes[0].set_title("Mean Absolute Error ($)")
axes[0].set_ylabel("MAE ($)")
axes[0].tick_params(axis="x", rotation=30)

axes[1].bar(error_by_range["PriceRange"],
            error_by_range["MeanPctError"], color=colors)
axes[1].set_title("Mean % Error")
axes[1].set_ylabel("MAPE (%)")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig("../reports/figures/error_by_price_range.png", dpi=150)
plt.show()


📊 Error Analysis by Price Range:
PriceRange  MeanAbsError  MeanPctError  Count
  Very Low  15318.828935      8.745434   2142
       Low  16363.755519      5.964560  10359
    Medium  22614.799812      5.721381   3847
      High  38197.707317      7.206100    451
 Very High 116460.090909     16.709923     33


In [7]:
# SHAP Values (Model Interpretability)
try:
    import shap
    print("🔄 Computing SHAP values...")

    explainer   = shap.Explainer(model)
    shap_values = explainer(X_test[:200])

    # Summary Plot
    plt.figure(figsize=(10, 8))
    shap.summary_plot(shap_values, X_test[:200],
                      show=False, plot_size=(10, 8))
    plt.title("SHAP Feature Importance", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig("../reports/figures/shap_summary.png", dpi=150)
    plt.show()
    print("✅ SHAP analysis complete.")

except ImportError:
    print("⚠️  SHAP not installed. Run: pip install shap")
    print("   Skipping SHAP analysis.")

⚠️  SHAP not installed. Run: pip install shap
   Skipping SHAP analysis.


In [8]:
# Prediction Confidence Intervals
# Bootstrap confidence intervals
n_bootstrap = 100
predictions_bootstrap = []

for _ in range(n_bootstrap):
    idx = np.random.choice(len(X_test), len(X_test), replace=True)
    X_boot = X_test.iloc[idx]
    y_boot = model.predict(X_boot)
    predictions_bootstrap.append(y_boot)

pred_array = np.array(predictions_bootstrap)
pred_mean  = pred_array.mean(axis=0)
pred_lower = np.percentile(pred_array, 5, axis=0)
pred_upper = np.percentile(pred_array, 95, axis=0)

# Plot for first 50 samples
fig, ax = plt.subplots(figsize=(14, 6))
idx_range = range(50)
ax.plot(idx_range, y_test.values[:50], "o-",
        color="#1E293B", label="Actual", markersize=4)
ax.plot(idx_range, pred_mean[:50], "s--",
        color="#2563EB", label="Predicted (Mean)", markersize=4)
ax.fill_between(idx_range, pred_lower[:50], pred_upper[:50],
                alpha=0.25, color="#2563EB", label="90% CI")

ax.set_xlabel("Sample Index")
ax.set_ylabel("House Price ($)")
ax.set_title("Predictions with Bootstrap Confidence Intervals (First 50 Samples)",
             fontsize=12, fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig("../reports/figures/prediction_confidence.png", dpi=150)
plt.show()

In [9]:
# ── Cell 9: Final Summary ─────────────────────────────────────
print("\n" + "=" * 60)
print("       📊 FINAL EVALUATION SUMMARY")
print("=" * 60)
print(f"  Model       : XGBoost Regressor (Tuned)")
print(f"  R² Score    : {r2_score(y_test, y_pred):.4f}")
print(f"  RMSE        : ${np.sqrt(mean_squared_error(y_test, y_pred)):,.2f}")
print(f"  MAE         : ${mean_absolute_error(y_test, y_pred):,.2f}")
print(f"  MAPE        : {mean_absolute_percentage_error(y_test, y_pred)*100:.2f}%")
print("-" * 60)
threshold_r2 = 0.85
final_r2 = r2_score(y_test, y_pred)
status = "✅ PASS" if final_r2 >= threshold_r2 else "❌ FAIL"
print(f"  Threshold Check (R² ≥ {threshold_r2}): {status}")
print("=" * 60)


       📊 FINAL EVALUATION SUMMARY
  Model       : XGBoost Regressor (Tuned)
  R² Score    : 0.9102
  RMSE        : $25,174.48
  MAE         : $18,440.74
  MAPE        : 6.32%
------------------------------------------------------------
  Threshold Check (R² ≥ 0.85): ✅ PASS
